# EdgeGuard PPE model — train & export (Google Colab, free T4 GPU)
Trains **YOLO11s** on the Ultralytics *Construction-PPE* dataset, which has explicit `helmet / no_helmet` **and** `gloves / no_gloves` classes, then exports ONNX in the exact format EdgeGuard's browser worker expects (`[1,3,640,640]` → `[1,16,8400]`).

**Runtime → Change runtime type → T4 GPU**, then Run all (~40–60 min).

For reliable *webcam close-up* results, also add 100–300 photos of yourself / your team in front of the actual camera (with and without helmet/gloves) — see step 2.

In [ ]:
!pip -q install ultralytics onnx onnxslim
import ultralytics; ultralytics.checks()

## 2. (Strongly recommended) add your own webcam images
Public PPE datasets are mostly CCTV/site photos taken from a distance; a person 0.5–1 m from a laptop camera looks very different. Capture frames from the EdgeGuard camera, label them in Roboflow/CVAT using **the same 12 class names** below, export as *YOLO* format, and upload a zip with `images/` and `labels/` folders. Set `OWN_ZIP` to its path, otherwise leave it empty.

In [ ]:
OWN_ZIP = ''   # e.g. '/content/my_webcam_ppe.zip'
import os, zipfile, glob, shutil
from ultralytics.utils import DATASETS_DIR
from ultralytics.data.utils import check_det_dataset
data = check_det_dataset('construction-ppe.yaml')   # downloads ~650 MB
print(data['names'])
if OWN_ZIP:
    root = '/content/own'; zipfile.ZipFile(OWN_ZIP).extractall(root)
    train_img = data['train']; train_lbl = train_img.replace('images','labels')
    for img in glob.glob(f'{root}/**/images/*', recursive=True):
        lbl = os.path.splitext(img.replace('/images/','/labels/'))[0] + '.txt'
        if os.path.exists(lbl):
            for _ in range(3):   # oversample your own domain x3
                k = f'own{_}_' + os.path.basename(img)
                shutil.copy(img, f'{train_img}/{k}')
                shutil.copy(lbl, f'{train_lbl}/{os.path.splitext(k)[0]}.txt')
    print('added own images')

## 3. Train
Augmentations are tuned toward close-up webcam frames (larger scale jitter, flips, HSV for indoor lighting).

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(data='construction-ppe.yaml', epochs=120, imgsz=640, batch=16, patience=30,
            scale=0.7, degrees=5, fliplr=0.5, hsv_v=0.5, mosaic=1.0, close_mosaic=15,
            mixup=0.1, cos_lr=True, name='edgeguard_ppe', exist_ok=True)
m = model.val()
names = model.names
for i,n in names.items():
    print(f'{n:12s} mAP50={m.box.maps[i]:.3f}')

## 4. Export ONNX for the browser (no NMS inside the graph — EdgeGuard does NMS itself)

In [ ]:
best = YOLO('runs/detect/edgeguard_ppe/weights/best.pt')
path = best.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False, nms=False)
import onnxruntime as ort, numpy as np, hashlib, shutil
s = ort.InferenceSession(path); o = s.run(None, {s.get_inputs()[0].name: np.zeros((1,3,640,640),np.float32)})[0]
assert o.shape == (1, 16, 8400), o.shape
shutil.copy(path, '/content/edgeguard-ppe-yolo11s.onnx')
print('sha256:', hashlib.sha256(open('/content/edgeguard-ppe-yolo11s.onnx','rb').read()).hexdigest())

## 5. Quick webcam-style sanity check
Upload a photo of yourself (with and without gloves/helmet) and look at the boxes.

In [ ]:
from google.colab import files
for f in files.upload():
    r = best.predict(f, conf=0.35)[0]; r.show()
    print([(best.names[int(c)], round(float(p),2)) for c,p in zip(r.boxes.cls, r.boxes.conf)])

## 6. Publish to EdgeGuard
1. Download `/content/edgeguard-ppe-yolo11s.onnx`.
2. GitHub → EdgeGuard → **Releases → Draft new release**, tag **`ppe-model-v1`**, attach the file (keep that exact name).
3. Paste the printed sha256 into `ppe-model.json` → `models.edgeguard-ppe-yolo11s-v1.sha256`.
4. Redeploy. The build downloads, verifies and hosts the model; browsers pick it up automatically (new cache key).

Alternatively, place the file at `.cache/edgeguard-ppe-yolo11s-v1.onnx` locally before `npm run build`.